# Wind API Usage (WindPy)

- w.wsd(): 日频时间序列
- w.wss(): 截面快照 
- w.wsi(): 分钟级序列 
- w.wst(): Tick序列 
- w.wset(): 数据集 (成分股/行业) 
- w.edb(): 宏观经济库 
- w.tdays(): 交易日序列 

In [ ]:
from WindPy import w
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

w.start()

## 1. w.wsd() — daily
w.wsd(codes, fields, begin, end, options): PriceAdj: F=前复权, B=后复权

In [ ]:
# 单股OHLCV
res = w.wsd("600519.SH", "open,high,low,close,volume,amt", "2024-01-01", "2024-12-31", "PriceAdj=F")
df = pd.DataFrame(res.Data, index=res.Fields, columns=res.Times).T
# display(df)

In [ ]:
# 多股收盘价
codes = "600519.SH,000858.SZ,000001.SZ,601318.SH"
res = w.wsd(codes, "close", "2024-06-01", "2024-12-31", "PriceAdj=F")
df = pd.DataFrame(res.Data, index=codes.split(","), columns=res.Times).T
# display(df)

In [ ]:
# 财务指标
res = w.wsd("600519.SH", "pe_ttm,pb_lf,roe_ttm2", "2024-01-01", "2024-12-31")
df = pd.DataFrame(res.Data, index=res.Fields, columns=res.Times).T
# display(df)

## 2. w.wss() — 截面快照
w.wss(codes, fields, options)

In [ ]:
codes = "600519.SH,000858.SZ,000001.SZ,601318.SH,300750.SZ"
res = w.wss(codes, "sec_name,mkt_cap_ard,pe_ttm,pb_lf,industry_sw", f"tradeDate={datetime.now().strftime('%Y%m%d')}")
df = pd.DataFrame(res.Data, index=res.Fields, columns=codes.split(",")).T
# display(df)

In [ ]:
# 财报截面
res = w.wss(codes, "tot_oper_rev,net_profit_is,roe_avg", "rptDate=20240630;rptType=1")
df = pd.DataFrame(res.Data, index=res.Fields, columns=codes.split(",")).T
# display(df)

## 3. w.wsi() — minute-level K line
w.wsi(codes, fields, begin, end, "BarSize=5"): 1/5/15/30/60 min

In [ ]:
res = w.wsi("600519.SH", "open,high,low,close,volume", "2024-12-20 09:30:00", "2024-12-20 15:00:00", "BarSize=5")
df = pd.DataFrame(res.Data, index=res.Fields, columns=res.Times).T
# display(df)

## 4. w.wst() — Tick data

In [ ]:
res = w.wst("600519.SH", "last,volume,amt,bid1,ask1,bsize1,asize1","2024-12-20 09:30:00", "2024-12-20 09:35:00")
df = pd.DataFrame(res.Data, index=res.Fields, columns=res.Times).T
# display(df)

## 5. w.wset() — 数据集 (成分股/行业)

In [ ]:
# 沪深300成分股
res = w.wset("indexconstituent", f"date={datetime.now().strftime('%Y-%m-%d')};windcode=000300.SH")
df = pd.DataFrame(res.Data, index=res.Fields).T
# print(f"成分股数量:", len(df))
# display(df)

In [ ]:
# 全部A股 (sectorid: a001010100000000=全A, a001010r00000000=科创板, 1000008892000000=沪深300)
res = w.wset("sectorconstituent", f"date={datetime.now().strftime('%Y-%m-%d')};sectorid=a001010100000000")
df = pd.DataFrame(res.Data, index=res.Fields).T
# print(f"全部A股: {len(df)}")
# display(df)

## 6. w.edb() — macros

In [ ]:
edb_codes = {"M0000612": "GDP同比", "M0000616": "CPI同比", "M0017126": "PMI", "M0001383": "M2同比", "M0009808": "社融当月值"}
res = w.edb(",".join(edb_codes.keys()), "2023-01-01", "2024-12-31")
df = pd.DataFrame(res.Data, index=list(edb_codes.values()), columns=res.Times).T
# display(df.dropna(how='all'))

In [ ]:
us_edb = {"G0000891": "联邦基金利率", "G0000886": "美10Y国债", "G0000903": "美CPI同比"}
res = w.edb(",".join(us_edb.keys()), "2023-01-01", "2024-12-31")
df = pd.DataFrame(res.Data, index=list(us_edb.values()), columns=res.Times).T
# display(df.dropna(how='all'))

## 7. 交易日tools

In [ ]:
res = w.tdays("2024-12-01", "2024-12-31", "")
# print([d.strftime("%Y-%m-%d") for d in res.Data[0]])

In [ ]:
res = w.tdaysoffset(-10, datetime.now().strftime("%Y-%m-%d"), "")
# print(f"10交易日前: {res.Data[0][0].strftime('%Y-%m-%d')}")

res = w.tdayscount("2024-01-01", "2024-12-31", "")
# print(f"2024交易日数: {res.Data[0][0]}")

## 8. 工具函数封装

In [ ]:
def wind_wsd_to_df(codes, fields, start, end, options=""):
    res = w.wsd(codes, fields, start, end, options)
    if res.ErrorCode != 0: raise RuntimeError(f"Wind Error {res.ErrorCode}")
    code_list = codes.split(",") if "," in codes else [codes]
    field_list = fields.split(",") if "," in fields else [fields]
    if len(code_list) == 1:
        return pd.DataFrame(res.Data, index=[f.upper() for f in field_list], columns=res.Times).T
    return pd.DataFrame(res.Data, index=code_list, columns=res.Times).T

def wind_wss_to_df(codes, fields, options=""):
    res = w.wss(codes, fields, options)
    if res.ErrorCode != 0: raise RuntimeError(f"Wind Error {res.ErrorCode}")
    return pd.DataFrame(res.Data, index=res.Fields, columns=codes.split(",")).T

def wind_wset_to_df(tableName, options=""):
    res = w.wset(tableName, options)
    if res.ErrorCode != 0: raise RuntimeError(f"Wind Error {res.ErrorCode}")
    return pd.DataFrame(res.Data, index=res.Fields).T

def wind_edb_to_df(codes_dict, start, end):
    res = w.edb(",".join(codes_dict.keys()), start, end)
    if res.ErrorCode != 0: raise RuntimeError(f"Wind Error {res.ErrorCode}")
    return pd.DataFrame(res.Data, index=list(codes_dict.values()), columns=res.Times).T.dropna(how='all')

In [ ]:
# display(wind_wsd_to_df("600519.SH", "close,volume", "2024-01-01", "2024-12-31", "PriceAdj=F"))
# display(wind_wss_to_df("600519.SH,000858.SZ", "sec_name,pe_ttm,pb_lf", f"tradeDate={datetime.now().strftime('%Y%m%d')}"))
# display(wind_wset_to_df("indexconstituent", f"date={datetime.now().strftime('%Y-%m-%d')};windcode=000300.SH"))

## 9. 批量下载

In [ ]:
import os
from tqdm import tqdm

def batch_download_daily(code_list, fields, start, end, save_dir="../data_preparation/database/PV", options="PriceAdj=F"):
    os.makedirs(save_dir, exist_ok=True)
    failed = []
    for code in tqdm(code_list):
        try:
            df = wind_wsd_to_df(code, fields, start, end, options)
            df.to_parquet(os.path.join(save_dir, f"{code.replace('.','_')}.parquet"))
        except Exception as e:
            failed.append((code, str(e)))
    print(f"Done. Success: {len(code_list)-len(failed)}, Failed: {len(failed)}")
    return failed

# hs300 = wind_wset_to_df("indexconstituent", "date=2024-12-31;windcode=000300.SH")
# batch_download_daily(hs300.iloc[:,1].tolist(), "open,high,low,close,volume,amt", "2020-01-01", "2024-12-31")

In [ ]:
w.close()